In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.utils import resample
from tqdm import tqdm

### Extracting dataset of images and labels

In [2]:
data_folder = Path('data')
results = pd.DataFrame(columns=['x','t','label'])
pattern = re.compile(r'ID-(\d+)_time-(\d+)_From(\d+)ParamSweep_Data\.csv')
for file in tqdm(data_folder.glob('*.csv'), desc="Progress:"):
    match = pattern.match(file.name)
    if match:
        x_str, t_str, N_str = match.groups()
        if int(N_str) == 2:
            x = int(x_str)
            t = int(t_str)
            filename = f'ID-{x}_time-{t}_From2ParamSweep_Data.csv'
            file_path = data_folder / filename
            if file_path.exists():
                df = pd.read_csv(file_path)
                M_phenotypes = df.loc[df['celltypes'] == 'Macrophage', 'phenotypes']
                N = len(M_phenotypes)
                if N == 0:
                    continue
                count_type_0 = (M_phenotypes <= 0.5).sum()
                count_type_1 = N - count_type_0
                label = 0 if count_type_0 > count_type_1 else 1
            results.loc[len(results)] = [x, t, label]
results = results.sort_values(by=['x', 't'], ascending=[True, True])

Progress:: 919it [00:05, 153.94it/s]


In [3]:
results['label'].value_counts()

label
0    673
1    246
Name: count, dtype: int64

In [4]:
results.to_csv('files/xt.csv',index=False)